# 1. Imports

In [1]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [2]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [3]:
# Criação da sessão Spark local
spark = SparkSession.builder.master("local[*]").appName("feature_engineering").getOrCreate()

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [4]:
# competition_id = 1 (Premier League)
# season = 2022-2023
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [5]:
#df_events.select('homePlayers', 'awayPlayers', 'balls').first()

In [6]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        StructField("visibility", StringType(), True),
        StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    'competitionId',
    'season', # dps mudar pra seasonId se necessário
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [7]:
#df_events.filter(F.size("homePlayers_parsed") == 0).show()

In [8]:
#df_events.filter(F.size("awayPlayers_parsed") == 0).show()

In [9]:
#df_events.filter(F.size("balls_parsed") == 0).show()

In [10]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [11]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

#df_games_raw.show()

In [12]:
df_games_raw.select('venueType').distinct().show()

+-------------+
|    venueType|
+-------------+
|      NEUTRAL|
|OPPONENT_HOME|
|    TEAM_HOME|
+-------------+



In [13]:
df_games_raw.filter(F.col('venueType') == 'NEUTRAL').select('gameId', 'date', 'season', 'venueType', '`team.name`', '`opponentTeam.name`', '`stadium.name`').sort('date').show(truncate=False)

+------+----------+---------+---------+----------------------+-----------------------+-------------+
|gameId|date      |season   |venueType|team.name             |opponentTeam.name      |stadium.name |
+------+----------+---------+---------+----------------------+-----------------------+-------------+
|4443  |2022-08-06|2022-2023|NEUTRAL  |Chelsea               |Everton                |Goodison Park|
|4458  |2022-08-20|2022-2023|NEUTRAL  |Everton               |Nottingham Forest      |Goodison Park|
|4490  |2022-09-03|2022-2023|NEUTRAL  |Everton               |Liverpool              |Goodison Park|
|4510  |2022-09-18|2022-2023|NEUTRAL  |Everton               |West Ham               |Goodison Park|
|4531  |2022-10-09|2022-2023|NEUTRAL  |Everton               |Manchester United      |Goodison Park|
|4558  |2022-10-22|2022-2023|NEUTRAL  |Crystal Palace        |Everton                |Goodison Park|
|4578  |2022-11-05|2022-2023|NEUTRAL  |Everton               |Leicester City         |Goodi

- Apesar de Goodison Park teoricamente ser estádio do Everton e se tratarem de todos os jogos do Everton, o venueType é neutro. Por conta disso, não irei considerar essas partidas.
- Caso necessário usá-las futuramente, podemos ver como ficam os eventos de home e away e se seguem essa estrutura acima mesmo o estádio sendo neutro. Uma sugestão pode ser considerar sempre Everton como casa, mas teria que ver se os eventos de posse ficam de acordo.

In [14]:
df_games_raw = df_games_raw.filter(F.col('venueType').isin(['TEAM_HOME', 'OPPONENT_HOME']))

In [15]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw
    .withColumns({
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.id`")).otherwise(F.col("`opponentTeam.id`")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.name`")).otherwise(F.col("`opponentTeam.name`")),
        
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.id`")).otherwise(F.col("`team.id`")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.name`")).otherwise(F.col("`team.name`")),
    })
    .select(
        'gameId', 
        'date',
        'season',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'),
        'venueType',
        'homeTeamId',
        'homeTeamName',
        'opponentTeamId',
        'opponentTeamName',
        F.col('teamStartSide').alias('homeTeamStartSide'),
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

# df_games.show(5)

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [16]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events
    .join(
        df_games.drop('season', 'competitionId', 'competitionName'), 
        on = "gameId", 
        how='left'
    )
    # filtro para remover os jogos que tinha mandante neutro (venueType == NEUTRAL)
    .filter(~F.col('date').isNull())
)

df_games_events = (
    df_games_events
    .withColumn(
        'homeTeamAttackDirection',
            F.when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Right')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Left')), 
                'Left'
            )
            .when(
                ((F.col('period') == 1) & (F.col('homeTeamStartSide') == 'Left')) | 
                ((F.col('period') == 2) & (F.col('homeTeamStartSide') == 'Right')), 
                'Right'
            )
        )
    .withColumn(
            'awayTeamAttackDirection',
            F.when(F.col('homeTeamAttackDirection') == 'Right', 'Left')
            .when(F.col('homeTeamAttackDirection') == 'Left', 'Right')
        )
    .drop('homeTeamStartSide')
)

df_games_events.show(5)

+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+-------------+---------------+-----------+-------------+--------------------+--------------------+--------------------+----------+---------+----------+---------------+--------------+----------------+----------------+-------------+------------+-----------------------+-----------------------+
|gameId|competitionId|   season|             eventId|   eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|eventPlayerName|eventTeamId|eventTeamName|  homePlayers_parsed|  awayPlayers_parsed|        balls_parsed|      date|venueType|homeTeamId|   homeTeamName|opponentTeamId|opponentTeamName|     stadiumName|stadiumLength|stadiumWidth|homeTeamAttackDirection|awayTeamAttackDirection|
+------+-------------+---------+--------------------+------------+--------------------+------+-----------------------+--------------

In [17]:
df_games_events.groupby('period').count().show()

+------+------+
|period| count|
+------+------+
|     1|456103|
|     2|442797|
+------+------+



In [18]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

+--------+------+
|homeTeam| count|
+--------+------+
|    NULL|  6538|
|    true|451847|
|   false|440515|
+--------+------+



In [19]:
# window function pra criação do Id de posse
w_pos = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId"
    )
    .orderBy("startGameClock")
)

df_games_events = (
    df_games_events
    .filter(
        # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
        (F.col('period').isin([1,2])) &
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse
    .withColumn(
        "possession_id",
        F.sum(
            F.when(
                F.col("homeTeam") != F.lag("homeTeam").over(w_pos), 1
            ).otherwise(0)
        ).over(
            w_pos.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

## Normalização do ataque sempre pra direita

In [20]:
# time com a posse está atacando e time sem está defendendo

df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

#df_games_events_tracking.show()

In [21]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            p["visibility"].alias("visibility"),
            p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        'attackingDirection',
        'need_side_revert'
        )
)

In [22]:
def euclidean_dist(x1, y1, x2, y2):
    return F.sqrt(
        F.pow(x1 - x2, 2) +
        F.pow(y1 - y2, 2)
    )

In [23]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

# Menor distância euclidiana entre os escanteios do lado esquerdo até a bola
progression_distance = F.round(
    F.least(
        euclidean_dist(ball_x, ball_y, left_x, top_y),
        euclidean_dist(ball_x, ball_y, left_x, bottom_y)
    ), 2)

## Criação das componentes de ameaça

- Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios do time com a posse até a bola) 
    - Hipótese: Quanto mais o jogador com posse percorrer o campo com a bola na direção do gol, maior a ameaça de gol por estar mais próximo dele.
    - Relação: Diretamente proporcional
- Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
    - Hipótese: Quanto MAIS jogadores dos dois times entre a bola e gol, MENOR a ameaça de gol por haver maior possibilidade de alguma ação defensiva e também por haver chances de um possível chute ser bloqueado.
    - Relação: Inversamente proporcional
- Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)
    - Hipótese: Quanto MAIOR a vantagem numérica do ataque em relação à defesa, MAIOR a ameaça de gol por ter maiores chance de ações ofensivas e menores chances de ações defensivas
    - Relação: Diretamente proporcional

In [24]:
df_games_events_players_ball_goal = (
    df_games_events_tracking_norm
     .withColumns({
        # Quantidade de jogadores do time mandante entre o gol esquerdo e a bola
        'attackers_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('attackingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        ),
        'defenders_between_ball_goal': (
            F.size(
                F.filter(
                    F.col('defendingPlayersNorm'),
                    lambda p: (
                        # jogadores no eixo x entre a bola e o gol direito
                        (p['x'] <= right_x) &
                        (p['x'] >= ball_x)
                    )
                )
            )
        )
    })
)

df_games_events_players_ball_goal = (
    df_games_events_players_ball_goal.withColumns({
        # Progressão em campo em direção ao gol defendido
        'progression_distance': progression_distance,
        
        # Quantidade absoluta de jogadores entre a bola e o gol
        'total_players_between_ball_goal': F.col('attackers_between_ball_goal') + F.col('defenders_between_ball_goal'),
        
        # Vantagem numérica do ataque em relação à defesa
        'atk_def_advantage_between_ball_goal': F.col('attackers_between_ball_goal') - F.col('defenders_between_ball_goal')
    })
)

# df_games_events_players_ball_goal.show()

### Criação da Ameaça pela Média das 3 componentes normalizadas com Min-Max

In [25]:
# Obtém mínimos e máximos das componentes
stats = (
    df_games_events_players_ball_goal
    .agg(
        F.min("progression_distance").alias("min_pd"),
        F.max("progression_distance").alias("max_pd"),

        F.min("total_players_between_ball_goal").alias("min_tp"),
        F.max("total_players_between_ball_goal").alias("max_tp"),

        F.min("atk_def_advantage_between_ball_goal").alias("min_adv"),
        F.max("atk_def_advantage_between_ball_goal").alias("max_adv")
    )
    .first()
)

df_games_events_players_ball_goal_norm = (
    df_games_events_players_ball_goal
    # Componentes normalizadas [0,1]
    .withColumns({
        # Progressão em campo em direção ao gol defendido
        "progression_distance_norm":
        F.round((F.col("progression_distance") - F.lit(stats["min_pd"])) / F.lit(stats["max_pd"] - stats["min_pd"]), 3),

        # Quantidade absoluta de jogadores entre a bola e o gol (1 - minmax por ser inversamente proporcional)
        "total_players_between_ball_goal_norm": F.round(1 - 
        (F.col("total_players_between_ball_goal") - F.lit(stats["min_tp"])) / F.lit(stats["max_tp"] - stats["min_tp"]), 3), 
        
        # Vantagem numérica do ataque em relação à defesa
        "atk_def_advantage_between_ball_goal_norm":
        F.round((F.col("atk_def_advantage_between_ball_goal") - F.lit(stats["min_adv"])) / F.lit(stats["max_adv"] - stats["min_adv"]), 3),

        # Threat score = média das 3 componentes
        "threat_score": F.round((
            F.col("progression_distance_norm") + F.col("total_players_between_ball_goal_norm") + F.col("atk_def_advantage_between_ball_goal_norm")
        ) / F.lit(3.0), 3)

    })
)

In [ ]:
# window function por competição-temporada-jogo-time com posse
w = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId",
        "homeTeam"
    )
    .orderBy("startGameClock")
)

df_threat_final = (
    df_games_events_players_ball_goal_norm
    .withColumn(
        "threat_score_delta",
        F.round(F.coalesce(F.col("threat_score") - F.lag("threat_score").over(w), F.lit(0.0)), 3))
)

#df_threat_final.show()

+------+-------------+---------+--------------------+---------+--------------------+------+-----------------------+--------------+--------+-------------+------------------+-----------+-------------+----------+-------------+----------+--------------+--------------+----------------+-------------+-------------+------------+-------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+--------------------+-------------------------------+-----------------------------------+-------------------------+------------------------------------+----------------------------------------+------------+------------------+
|gameId|competitionId|   season|             eventId|eventType|eventTypeDescription|period|startFormattedGameClock|startGameClock|homeTeam|eventPlayerId|   eventPlayerName|eventTeamId|eventTeamName|      date|    venueType|homeTeamId|  homeTeamName|opponentTeamId|opponentTeamName|  stadiumName|stadiumLength|stadiumWid

### Validação do threat_score criado

In [27]:
filtered_columns = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'eventType',
    'period',
    'startFormattedGameClock',
    'startGameClock',
    'homeTeam',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'threat_score'
]

df_threat_filtrado = (
    df_threat_final.select(filtered_columns)
)

df_threat_filtrado.show()

+------+-------------+---------+----------+--------------------+------------+------+-----------------------+--------------+--------+--------------+----------------+-------------+------------+
|gameId|competitionId|   season|      date|             eventId|   eventType|period|startFormattedGameClock|startGameClock|homeTeam|  homeTeamName|opponentTeamName|possession_id|threat_score|
+------+-------------+---------+----------+--------------------+------------+------+-----------------------+--------------+--------+--------------+----------------+-------------+------------+
|  4436|            1|2022-2023|2022-08-05|6963e29748b45b531...|FIRSTKICKOFF|     1|                  00:00|             0|    true|Crystal Palace|         Arsenal|            0|       0.646|
|  4436|            1|2022-2023|2022-08-05|6cbfa1cd4e950a89f...|          PA|     1|                  00:00|             0|    true|Crystal Palace|         Arsenal|            0|       0.646|
|  4436|            1|2022-2023|2022-08-

In [ ]:
'''df_avg_threat = (
    df_threat_filtrado
    .groupBy(
        "gameId",
        "competitionId",
        "season",
        "date",
        'homeTeamName', 
        'opponentTeamName'
    )
    .agg(
        # ameaça média sofrida do time mandante
        F.round(F.mean(F.when(F.col("homeTeam"), F.col("threat_score"))), 3).alias("avg_threat_score_H"),

        # ameaça média sofrida do time visitante
        F.round(F.mean(F.when(~F.col("homeTeam"), F.col("threat_score"))), 3).alias("avg_threat_score_A")
    )
)

df_avg_threat.show()'''

+------+-------------+---------+----------+--------------------+--------------------+------------------+------------------+
|gameId|competitionId|   season|      date|        homeTeamName|    opponentTeamName|avg_threat_score_H|avg_threat_score_A|
+------+-------------+---------+----------+--------------------+--------------------+------------------+------------------+
|  4450|            1|2022-2023|2022-08-14|             Chelsea|   Tottenham Hotspur|             0.433|             0.377|
|  4436|            1|2022-2023|2022-08-05|      Crystal Palace|             Arsenal|             0.597|             0.579|
|  4441|            1|2022-2023|2022-08-06|    Newcastle United|   Nottingham Forest|             0.454|             0.368|
|  4460|            1|2022-2023|2022-08-21|        Leeds United|             Chelsea|             0.567|             0.572|
|  4465|            1|2022-2023|2022-08-21|            West Ham|Brighton & Hove A...|             0.582|             0.571|
|  4447|

In [66]:
df_avg_threat = (
    df_threat_filtrado
    .groupBy(
        "gameId",
        "competitionId",
        "season",
        "date",
        'homeTeamName', 
        'opponentTeamName',
        "homeTeam",
    )
    .agg(
        F.round(F.mean(F.col("threat_score")), 3).alias("avg_threat_score"),
    )
)

df_avg_threat.show()

+------+-------------+---------+----------+--------------------+--------------------+--------+----------------+
|gameId|competitionId|   season|      date|        homeTeamName|    opponentTeamName|homeTeam|avg_threat_score|
+------+-------------+---------+----------+--------------------+--------------------+--------+----------------+
|  4469|            1|2022-2023|2022-08-27|Brighton & Hove A...|        Leeds United|    true|            0.43|
|  4449|            1|2022-2023|2022-08-13|Brighton & Hove A...|    Newcastle United|   false|           0.411|
|  4460|            1|2022-2023|2022-08-21|        Leeds United|             Chelsea|    true|           0.567|
|  4438|            1|2022-2023|2022-08-06|     AFC Bournemouth|         Aston Villa|   false|           0.402|
|  4437|            1|2022-2023|2022-08-06|              Fulham|           Liverpool|   false|           0.389|
|  4457|            1|2022-2023|2022-08-20|      Crystal Palace|         Aston Villa|   false|          

In [29]:
pl_match_stats_22_23 = str(Path().resolve().parent.parent / "data" / "match_stats" / "PL_22_23.csv")

df_pl_match_stats_22_23 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(pl_match_stats_22_23, sep=',')    

df_pl_match_stats_22_23.show()

+---+----------+-------------------+--------------+--------------+----+----+---+----+----+---+------------+---+---+---+---+---+---+---+---+---+---+---+---+-----+-----+-----+----+----+----+----+----+----+----+-----+-----+----+----+----+----+----+----+----+----+----+-----+-----+-----+--------+--------+-----+-----+-------+-------+-------+-------+-----+-------+-------+----+----+------+------+------+------+------+------+------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+---------+---------+------+------+--------+--------+--------+--------+-----+--------+--------+-----+-----+-------+-------+-------+-------+
|Div|      Date|               Time|      HomeTeam|      AwayTeam|FTHG|FTAG|FTR|HTHG|HTAG|HTR|     Referee| HS| AS|HST|AST| HF| AF| HC| AC| HY| AY| HR| AR|B365H|B365D|B365A| BWH| BWD| BWA| IWH| IWD| IWA| PSH|  PSD|  PSA| WHH| WHD| WHA| VCH| VCD| VCA|MaxH|MaxD|MaxA| AvgH| AvgD| AvgA|B365>2.5|B365<2.5|P>2.5|P<2.5|Max>2.5|Max

In [34]:
df_pl_match_stats_22_23_filtrado = df_pl_match_stats_22_23.select(
    F.to_date(F.col("Date"), "dd/MM/yyyy").alias("date"),
    F.col('HomeTeam').alias('homeTeamName'),
    F.col('AwayTeam').alias('opponentTeamName'),
    'FTHG',
    'FTAG',
    'FTR',
    #'HTHG',
    #'HTAG',
    #'HTR',
    'HS',
    'AS',
    'HST',
    'AST',    
    'AvgH',
    'AvgA',
    'AvgD'
).sort('date')

In [48]:
team_name_mapping = {
    "Tottenham": "Tottenham Hotspur",
    "Brighton": "Brighton & Hove Albion",
    "Man City": "Manchester City",
    "Crystal Palace": "Crystal Palace",
    "Leicester": "Leicester City",
    "Aston Villa": "Aston Villa",
    "Bournemouth": "AFC Bournemouth",
    "Fulham": "Fulham",
    "West Ham": "West Ham",
    "Man United": "Manchester United",
    "Wolves": "Wolverhampton Wanderers",
    "Southampton": "Southampton",
    "Liverpool": "Liverpool",
    "Chelsea": "Chelsea",
    "Nott'm Forest": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Everton": "Everton",
    "Leeds": "Leeds United",
    "Arsenal": "Arsenal",
    "Brentford": "Brentford"
}

df_pl_match_stats_22_23_filtrado_mapped = (
    df_pl_match_stats_22_23_filtrado
    .replace(team_name_mapping, subset=["homeTeamName", "opponentTeamName"])
)

df_pl_match_stats_22_23_filtrado_mapped.show()

+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|      date|        homeTeamName|    opponentTeamName|FTHG|FTAG|FTR| HS| AS|HST|AST| AvgH| AvgA| AvgD|
+----------+--------------------+--------------------+----+----+---+---+---+---+---+-----+-----+-----+
|2022-08-05|      Crystal Palace|             Arsenal|   0|   2|  A| 10| 10|  2|  2| 4.39| 1.88| 3.59|
|2022-08-06|              Fulham|           Liverpool|   2|   2|  D|  9| 11|  3|  4|10.99| 1.28| 6.05|
|2022-08-06|     AFC Bournemouth|         Aston Villa|   2|   0|  H|  7| 15|  3|  2|  3.8| 2.04|  3.5|
|2022-08-06|        Leeds United|Wolverhampton Wan...|   2|   1|  H| 12| 15|  4|  6| 2.34| 3.18| 3.34|
|2022-08-06|    Newcastle United|   Nottingham Forest|   2|   0|  H| 23|  5| 10|  0| 1.67| 5.57|  3.8|
|2022-08-06|   Tottenham Hotspur|         Southampton|   4|   1|  H| 18| 10|  8|  2| 1.36| 8.64| 5.27|
|2022-08-06|             Everton|             Chelsea|   0|   1|  A|  8| 

In [67]:
df_avg_threat_match_stats = (
    df_avg_threat.join(
        df_pl_match_stats_22_23_filtrado_mapped,
        on= ['date', 'homeTeamName', 'opponentTeamName'],
        how='left'
    )
)

df_avg_threat_match_stats.show()

+----------+--------------------+--------------------+------+-------------+---------+--------+----------------+----+----+---+---+---+---+---+-----+-----+-----+
|      date|        homeTeamName|    opponentTeamName|gameId|competitionId|   season|homeTeam|avg_threat_score|FTHG|FTAG|FTR| HS| AS|HST|AST| AvgH| AvgA| AvgD|
+----------+--------------------+--------------------+------+-------------+---------+--------+----------------+----+----+---+---+---+---+---+-----+-----+-----+
|2022-08-27|Brighton & Hove A...|        Leeds United|  4469|            1|2022-2023|    true|            0.43|   1|   0|  H| 13| 10|  4|  2| 1.91| 4.04| 3.73|
|2022-08-13|Brighton & Hove A...|    Newcastle United|  4449|            1|2022-2023|   false|           0.411|   0|   0|  D| 13|  4|  7|  1| 2.44|  3.1| 3.25|
|2022-08-21|        Leeds United|             Chelsea|  4460|            1|2022-2023|    true|           0.567|   3|   0|  H| 12| 14|  6|  3| 5.93| 1.55| 4.43|
|2022-08-06|     AFC Bournemouth|       

In [71]:
df_pl_match_stats_22_23_filtrado_mapped_home = df_avg_threat_match_stats.filter(
    F.col('homeTeam')
    ).select(
    "date",
    F.col("homeTeamName").alias("teamName"),
    "avg_threat_score",
    (F.col("FTR") == 'H').alias('win'),
    F.col("FTHG").alias("goals"),
    F.col("HS").alias("shots"),
    F.col("HST").alias("shots_target"),
    F.col("AvgH").alias("avg_win_odds")
)

df_pl_match_stats_22_23_filtrado_mapped_away = df_avg_threat_match_stats.filter(
    ~F.col('homeTeam')
    ).select(
    "date",
    F.col("opponentTeamName").alias("teamName"),
    "avg_threat_score",
    (F.col("FTR") == 'A').alias('win'),
    F.col("FTAG").alias("goals"),
    F.col("AS").alias("shots"),
    F.col("AST").alias("shots_target"),
    F.col("AvgA").alias("avg_win_odds")
)

df_team_match = df_pl_match_stats_22_23_filtrado_mapped_home.unionByName(df_pl_match_stats_22_23_filtrado_mapped_away)
df_team_match.show()

+----------+--------------------+----------------+-----+-----+-----+------------+------------+
|      date|            teamName|avg_threat_score|  win|goals|shots|shots_target|avg_win_odds|
+----------+--------------------+----------------+-----+-----+-----+------------+------------+
|2022-08-27|Brighton & Hove A...|            0.43| true|    1|   13|           4|        1.91|
|2022-08-21|        Leeds United|           0.567| true|    3|   12|           6|        5.93|
|2022-08-20|      Crystal Palace|           0.566| true|    3|   17|           9|        2.52|
|2022-08-13|     Manchester City|            0.57| true|    4|   19|           7|        1.07|
|2022-08-14|   Nottingham Forest|           0.427| true|    1|   13|           6|        4.21|
|2022-08-22|   Manchester United|            0.58| true|    2|   12|           4|        5.22|
|2022-08-07|      Leicester City|             0.6|false|    2|   14|           5|        2.03|
|2022-08-13|           Brentford|           0.446|

In [90]:
df_team_match_pd = df_team_match.toPandas()
df_team_match_pd.drop(['date', 'teamName'], axis=1).corr()

,avg_threat_score,win,goals,shots,shots_target,avg_win_odds
avg_threat_score,1.000000,0.001259,-0.023419,-0.011282,-0.022424,0.017894
win,0.001259,1.000000,0.620001,0.252019,0.385947,-0.287305
goals,-0.023419,0.620001,1.000000,0.330004,0.587279,-0.224993
shots,-0.011282,0.252019,0.330004,1.000000,0.695799,-0.421675
shots_target,-0.022424,0.385947,0.587279,0.695799,1.000000,-0.306992
avg_win_odds,0.017894,-0.287305,-0.224993,-0.421675,-0.306992,1.000000


In [ ]:
# Saldo do time da casa na temporada
df_avg_threat_teams_season = (
    df_team_match.groupby(
        'TeamName'
        )
    .agg(
        F.median(F.col('avg_threat_score')).alias('avg_threat_score'),
        
        F.round(F.mean(F.when(F.col("win"), 1).otherwise(0)), 2).alias("win_score"),
        
        F.sum(F.col('goals')).alias('total_goals'),
        F.round(F.mean(F.col('goals')), 2).alias('avg_goals'),
        
        F.sum(F.col('shots')).alias('total_shots'),
        F.round(F.mean(F.col('shots')), 2).alias('avg_shots'),
        
        F.sum(F.col('shots_target')).alias('total_shots_target'),
        F.round(F.mean(F.col('shots_target')), 2).alias('avg_shots_target'),
        
        F.round(F.median(F.col('avg_win_odds')), 2).alias('avg_win_odds'),       
    ).sort('win_score', ascending=False)
)

#df_avg_threat_teams_season.show()

+--------------------+----------------+---------+-----------+---------+-----------+---------+------------------+----------------+------------+
|            TeamName|avg_threat_score|win_score|total_goals|avg_goals|total_shots|avg_shots|total_shots_target|avg_shots_target|avg_win_odds|
+--------------------+----------------+---------+-----------+---------+-----------+---------+------------------+----------------+------------+
|     Manchester City|            0.56|     0.73|         91|     2.46|        594|    16.05|               215|            5.81|         1.3|
|             Arsenal|           0.438|      0.7|         88|     2.38|        578|    15.62|               201|            5.43|        1.59|
|   Manchester United|           0.556|     0.59|         56|     1.51|        580|    15.68|               210|            5.68|        1.89|
|           Liverpool|           0.553|     0.51|         75|     2.03|        582|    15.73|               204|            5.51|        1.54|

In [87]:
df_avg_threat_teams_season_pd = df_avg_threat_teams_season.toPandas()
df_avg_threat_teams_season_pd.drop('TeamName', axis=1).corr('spearman')

,avg_threat_score,win_score,total_goals,avg_goals,total_shots,avg_shots,total_shots_target,avg_shots_target,avg_win_odds
avg_threat_score,1.000000,-0.129811,0.000000,-0.003771,-0.066993,-0.065487,-0.091080,-0.105382,0.026346
win_score,-0.129811,1.000000,0.870469,0.860650,0.762913,0.744820,0.805883,0.791559,-0.782513
total_goals,0.000000,0.870469,1.000000,0.995472,0.755557,0.743504,0.885877,0.878344,-0.762337
avg_goals,-0.003771,0.860650,0.995472,1.000000,0.738984,0.732958,0.875331,0.875331,-0.753297
total_shots,-0.066993,0.762913,0.755557,0.738984,1.000000,0.995489,0.873684,0.861654,-0.902256
avg_shots,-0.065487,0.744820,0.743504,0.732958,0.995489,1.000000,0.867669,0.863158,-0.906767
total_shots_target,-0.091080,0.805883,0.885877,0.875331,0.873684,0.867669,1.000000,0.995489,-0.855639
avg_shots_target,-0.105382,0.791559,0.878344,0.875331,0.861654,0.863158,0.995489,1.000000,-0.851128
avg_win_odds,0.026346,-0.782513,-0.762337,-0.753297,-0.902256,-0.906767,-0.855639,-0.851128,1.000000
